In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from typing import Tuple

class SatelliteTX:
    """
    Satellite Transmitter Baseband Processor.

    This class handles the conversion of text data into a baseband
    complex signal ready for RF up-conversion or analog simulation.

    Attributes:
        symbol_rate (int): Number of symbols transmitted per second (Baud rate).
        sps (int): Samples per symbol (Oversampling factor).
        fs (int): System sampling rate in Hz.
        alpha (float): Roll-off factor for the Root Raised Cosine filter.
    """

    def __init__(self, symbol_rate: int = 10000, sps: int = 8, alpha: float = 0.35):
        self.symbol_rate = symbol_rate
        self.sps = sps
        self.fs = symbol_rate * sps
        self.alpha = alpha

    def text_to_bits(self, text: str) -> np.ndarray:
        """
        Convert an ASCII string into a stream of binary bits.
        Each character is represented by 8 bits (MSB first).
        """
        bits = []
        for char in text:
            bits.extend([int(b) for b in format(ord(char), '08b')])
        return np.array(bits)

    def bits_to_qpsk(self, bits: np.ndarray) -> np.ndarray:
        """
        Map a bit stream to complex QPSK symbols using Gray coding.
        """
        if len(bits) % 2 != 0:
            bits = np.append(bits, 0)
        reshaped = bits.reshape(-1, 2)
        symbols = []
        for b in reshaped:
            i = 1.0 if b[0] == 0 else -1.0
            q = 1.0 if b[1] == 0 else -1.0
            symbols.append(complex(i, q) / np.sqrt(2))
        return np.array(symbols)

    def _generate_rrc_taps(self, num_symbols: int = 10) -> np.ndarray:
        """Generate the impulse response (taps) for the Root Raised Cosine filter."""
        T = 1.0 / self.symbol_rate
        t = np.arange(-num_symbols * T, num_symbols * T + 1/self.fs, 1/self.fs)
        taps = np.zeros(len(t))
        for idx, ti in enumerate(t):
            if ti == 0.0:
                taps[idx] = 1.0 - self.alpha + (4 * self.alpha / np.pi)
            elif abs(ti) == T / (4 * self.alpha):
                taps[idx] = (self.alpha / np.sqrt(2)) * (((1 + 2/np.pi) * np.sin(np.pi/(4 * self.alpha))) + ((1 - 2/np.pi) * np.cos(np.pi/(4 * self.alpha))))
            else:
                num = np.sin(np.pi * ti / T * (1 - self.alpha)) + 4 * self.alpha * ti / T * np.cos(np.pi * ti / T * (1 + self.alpha))
                den = np.pi * ti / T * (1 - (4 * self.alpha * ti / T)**2)
                taps[idx] = num / den
        return taps / np.sqrt(np.sum(taps**2))

    def apply_rrc_filter(self, symbols: np.ndarray) -> np.ndarray:
        """Perform pulse shaping on the QPSK symbols."""
        upsampled = np.zeros(len(symbols) * self.sps, dtype=complex)
        upsampled[::self.sps] = symbols
        taps = self._generate_rrc_taps()
        tx_signal = np.convolve(upsampled, taps, mode='same')
        return tx_signal

    def export_to_csv(self, signal: np.ndarray, filename: str = "tx_baseband.csv") -> None:
        """
        Export the complex signal to a CSV file for Qucs Studio interaction.
        The columns are: Time (s), I (Real), Q (Imag).
        """
        time_axis = np.arange(len(signal)) / self.fs
        df = pd.DataFrame({
            'Time': time_axis,
            'I': signal.real,
            'Q': signal.imag
        })
        df.to_csv(filename, index=False)
        print(f"[System Info] Signal exported successfully to {filename}")

In [ ]:
# Finalizing Phase 1: Execution and File Export
tx = SatelliteTX(symbol_rate=10000, sps=8, alpha=0.35)
test_str = "Hello LEO Satellite!"

# DSP Chain execution
bits = tx.text_to_bits(test_str)
symbols = tx.bits_to_qpsk(bits)
signal = tx.apply_rrc_filter(symbols)

# Exporting the data for the 'Analog World' (Qucs Studio)
tx.export_to_csv(signal, "tx_baseband_signal.csv")

# Preview the first few rows of the data we just exported
import pandas as pd
display(pd.read_csv("tx_baseband_signal.csv").head())

[System Info] Signal exported successfully to tx_baseband_signal.csv


,Time,I,Q
0,0.000000,0.230747,-0.285737
1,0.000013,0.263801,-0.256537
2,0.000025,0.296669,-0.201592
3,0.000037,0.326546,-0.125003
4,0.000050,0.349387,-0.033721
